In [8]:
%pip install python-dotenv openai datasets math_verify tqdm torch aiolimiter

Note: you may need to restart the kernel to use updated packages.


In [9]:
from dotenv import load_dotenv

load_dotenv()

True

In [10]:
import logging

logging.basicConfig(level=logging.INFO)

In [ ]:
import os
from openai import AsyncOpenAI
from aiolimiter import AsyncLimiter
from asyncio import Semaphore
from math_verify import parse
from dataclasses import dataclass

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
MODEL_ID = "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning"

client = AsyncOpenAI(
	base_url="https://integrate.api.nvidia.com/v1",
	api_key=NVIDIA_API_KEY,
	max_retries=0,
)

limiter = AsyncLimiter(40)
semaphore = Semaphore(10)
async def create_completion(*args, **kwargs):
	while True:
		try:
			async with limiter:
				async with semaphore:
					return await client.chat.completions.create(*args, **kwargs)
		except:
			pass

@dataclass
class MyCompletionChoice:
	reasoning_content: str = ""
	content: str = ""

async def buffer_completion_stream(response):
	choices: list[MyCompletionChoice] = []
	async for chunk in response:
		for choice in chunk.choices:
			while len(choices) <= choice.index:
				choices.append(MyCompletionChoice())

			if delta := getattr(choice.delta, "reasoning_content", None):
				choices[choice.index].reasoning_content += delta

			if delta := getattr(choice.delta, "content", None):
				choices[choice.index].content += delta
	return choices

async def create_and_buffer_completion_stream(**kwargs):
	kwargs.pop("stream", None)
	stream = await create_completion(**kwargs, stream=True)
	return await buffer_completion_stream(stream)

prompt = "What is 13 times 17? Box your answer."
gold = "221"

choices = await create_and_buffer_completion_stream(
	model=MODEL_ID,
	messages=[{"role": "user", "content": prompt}],
	max_tokens=2**10,
    n=2,
	extra_body={
        "chat_template_kwargs": {"enable_thinking": True},
        "reasoning_budget": 2**10
    },
)

print("*"*20, "Prompt", "*"*20)
print(prompt)

parsed_gold = parse(gold) or [None]
for i, choice in enumerate(choices):
  parsed_answer = parse(choice.content) or [None]
  correct = parsed_gold[0] == parsed_answer[0]

  print("*"*20, f"Choice {i+1}: {parsed_answer[0]} ({'correct' if correct else 'incorrect'})", "*"*20)
  if correct:
    print(f"<think>{choice.reasoning_content}</think>{choice.content}")

INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"


******************** Prompt ********************
What is 13 times 17? Box your answer.
******************** Generation 1: 221 (correct) ********************
<think>The user asks: "What is 13 times 17? Box your answer." So we need to compute 13*17 = 221. Then we need to "box" the answer. Probably using a box like a LaTeX \boxed{221} or a textual box. Typically they want something like \boxed{221}. So answer: \boxed{221}. Ensure we follow instructions. There's no disallowed content. Provide answer.
</think>\[
\boxed{221}
\]
******************** Generation 2: 221 (correct) ********************
<think>The user asks: "What is 13 times 17? Box your answer."

We need to compute 13 * 17 = 221. Then "box" the answer. Usually that means using a box format, maybe like \boxed{221} or using a text box. The typical answer format in these tasks is to put the answer inside a box, like:

\[
\boxed{221}
\]

Or maybe a simple text box: [221</think>] or something. The instruction: "Box your answer." Likel

In [ ]:
from datasets import load_dataset

ds = load_dataset("open-r1/OpenR1-Math-220k", "default")
ds

INFO:httpx:HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/open-r1/OpenR1-Math-220k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/open-r1/OpenR1-Math-220k/e4e141ec9dea9f8326f4d347be56105859b2bd68/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/open-r1/OpenR1-Math-220k/resolve/e4e141ec9dea9f8326f4d347be56105859b2bd68/OpenR1-Math-220k.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/open-r1/OpenR1-Math-220k/open-r1/OpenR1-Math-220k.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/open-r1/OpenR1-Math-220k/revision/e4e141ec9dea9f8326f4d347be56105859b2bd68 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/open-r1/OpenR1-M

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/open-r1/OpenR1-Math-220k/resolve/e4e141ec9dea9f8326f4d347be56105859b2bd68/dataset_infos.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/open-r1/OpenR1-Math-220k/tree/e4e141ec9dea9f8326f4d347be56105859b2bd68/data?recursive=true&expand=false "HTTP/1.1 200 OK"


DatasetDict({
    train: Dataset({
        features: ['problem', 'solution', 'answer', 'problem_type', 'question_type', 'source', 'uuid', 'is_reasoning_complete', 'generations', 'correctness_math_verify', 'correctness_llama', 'finish_reasons', 'correctness_count', 'messages'],
        num_rows: 93733
    })
})

In [ ]:
from tqdm.contrib.logging import logging_redirect_tqdm
from tqdm.asyncio import tqdm_asyncio
from math_verify import verify
import torch
from datasets import Dataset

async def generate_dataset(prompts, golds, **kwargs):
	dataset_dict = {
		"prompt": [],
		"outputs": [],
		"advantages": [],
	}
	futures = []
	for prompt in prompts:
		futures.append(create_and_buffer_completion_stream(
			**kwargs,
			messages=[{"role": "user", "content": prompt}],
		))
	with logging_redirect_tqdm():
		completions = await tqdm_asyncio.gather(*futures, desc="Creating completions")
	for prompt, choices, gold in zip(prompts, completions, golds):
		gold = parse(gold)

		outputs = []
		rewards = []
		for choice in choices:
			answer = parse(choice.content)
			correct = verify(gold, answer)

			outputs.append(f"<think>{choice.reasoning_content}</think>{choice.content}")
			rewards.append(1.0 if correct else 0.0)
		rewards = torch.tensor(rewards, dtype=torch.float32)

		rewards_std = rewards.std()
		if rewards_std < 1e-5:
			advantages = torch.zeros_like(rewards)
		else:
			advantages = (rewards - rewards.mean()) / rewards_std

		dataset_dict["prompt"].append(prompt)
		dataset_dict["outputs"].append(outputs)
		dataset_dict["advantages"].append(advantages.tolist())
	return Dataset.from_dict(dataset_dict)

example_ds = await generate_dataset(
	prompts=[
		"Pick an random integer from 1 to 3. Don't pick 2. Box your answer.",
		"What is 8 times 3? Box your answer.",
	],
	golds=["3", "24"],
	model=MODEL_ID,
	max_tokens=2**10,
    n=8,
	extra_body={
        "chat_template_kwargs": {"enable_thinking": True},
        "reasoning_budget": 2**10,
    },
)
example_ds[:]

INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
Creating completions: 100%|██████████| 2/2 [01:33<00:00, 46.82s/it]


{'prompt': ["Pick an random integer from 1 to 3. Don't pick 2. Box your answer.",
  'What is 8 times 3? Box your answer.'],
 'outputs': [['<think>Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User asks: "Pick a random integer from 1 to 3. Don\'t pick 2. Box your answer."\n   - Constraints: \n     - Must pick an integer from 1 to 3 inclusive\n     - Must NOT pick 2\n     - Must box the answer (probably using markdown `**` or `[]` or similar, but I\'ll use `**` or a box format)\n\n2.  **Determine Valid Options:**\n   - Integers from 1 to 3: {1, 2, 3}\n   - Exclude 2: {1, 3}\n   - Since they said "random" but also "Don\'t pick 2", I need to choose randomly from the remaining valid options: 1 or 3.\n\n3.  **Make a Choice:**\n   - I\'ll pick one randomly. Let\'s say I choose 1. Or 3. It doesn\'t matter much, but I\'ll pick one. I\'ll go with 1, or maybe I\'ll actually randomize by choosing the first valid one, or I\'ll just pick 3. Actually, I\'ll pick 1. Wait, to be fair

In [ ]:
input_ds = ds["train"].shuffle().select(range(128))
output_ds = await generate_dataset(
	prompts=input_ds["problem"],
	golds=input_ds["answer"],
	model=MODEL_ID,
	max_tokens=2**15,
    n=64,
	extra_body={
        "chat_template_kwargs": {"enable_thinking": True},
        "reasoning_budget": 2**15,
    },
)
output_ds

INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 500 Internal Server Error"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 500 Internal Server Error"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 504 Gateway Timeout"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 504 Gateway Timeout"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 504 Gateway Timeout"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 504 Gateway Timeout"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 504 Gateway Timeout"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 504 Gateway Timeout"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/c